<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning--Models/blob/main/RAG_learning3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Module 1: The Coordinate-Based Extractor (LayoutExtractor).
!pip install pymupdf -q
import fitz
import urllib.request
#download attention pdf
pdf_url="https://arxiv.org/pdf/1706.03762.pdf"
local_pdf="attention.pdf"
urllib.request.urlretrieve(pdf_url,local_pdf)

class LayoutExtractor:
  def __init__(self,pdf_path):
    self.pdf_path=pdf_path

  def extract_page_sorted_blocks(self,page_num):
    doc=fitz.open(self.pdf_path)
    page=doc[page_num]
    rect=page.rect
    page_width=rect.width
    mid_x=page_width/2

    #get raw blocks(x0,y0,x1,y1,text,block_no,block_type)
    #block type text=0,img=1
    raw_blocks=page.get_text("blocks")

    full_width_blocks=[]
    left_column_blocks=[]
    right_column_blocks=[]

    for b in raw_blocks:
      x0,y0,x1,y1,text,block_no,block_type=b
      if block_type != 0:
        continue

      block_width=x1-x0
      if block_width >(page_width*0.7):
        full_width_blocks.append((y0,b))
      elif x1<mid_x+20: #belongs to left column element
        left_column_blocks.append((y0,b))
      else:
        right_column_blocks.append((y0,b))

    #sort each group vertically by y0 coordinate
    full_width_blocks.sort(key=lambda x:x[0])
    left_column_blocks.sort(key=lambda x:x[0])
    right_column_blocks.sort(key=lambda x:x[0])

    #reconstruct logical reading order
    #usually full width elements at top->left col->right col->

    sorted_blocks=[item[1] for item in full_width_blocks]+\
                  [item[1] for item in left_column_blocks]+\
                  [item[1] for item in right_column_blocks]

    return sorted_blocks

# Test the Extractor on Page 3 (where two-column layout starts)
extractor = LayoutExtractor(local_pdf)
page_3_blocks = extractor.extract_page_sorted_blocks(2) # 0-indexed, so page 3 is index 2

print(f"Successfully extracted and sorted {len(page_3_blocks)} blocks from Page 3!")
print("\n--- FIRST 3 SORTED BLOCKS ---\n")
for i in range(min(3, len(page_3_blocks))):
    print(f"Block {i+1}:\n{page_3_blocks[i][4].strip()}\n" + "-"*30)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.4 MB/s eta 0:00:00
Successfully extracted and sorted 8 blocks from Page 3!

--- FIRST 3 SORTED BLOCKS ---

Block 1:
3.1
Encoder and Decoder Stacks
------------------------------
Block 2:
3.2
Attention
------------------------------
Block 3:
3
------------------------------
